In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyreadstat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold
from sklearn.metrics import make_scorer, mean_absolute_error
from typing import Dict, Tuple, Union, List


In [7]:
#code/functions for preprocessing PISA Dataset 

def LFM_data(student_path, school_path, country):  #load + filter + merge data
    print(f"Loading student and school data...")
    student_df, _ = pyreadstat.read_sav(student_path, apply_value_formats=False) 
    school_df, _ = pyreadstat.read_sav(school_path, apply_value_formats=False)

    print(f"Filtering student and school data for country...")
    student_df = student_df[student_df["CNT"] == country]
    school_df = school_df[school_df["CNT"] == country][["CNTSCHID","RATCMP1","RATCMP2","EDUSHORT"]]

    print(f"Merging student + school data...")
    return student_df.merge(school_df, on="CNTSCHID", how="left")

def process_data(df, predictors, dv):
    print(f"Keeping predictors + dv, declaring x and y, and encoding gender...")
    df = df[predictors + [dv, "CNTSCHID"]]

    X = df.drop(columns = [dv, "CNTSCHID"]) #drop DV + School ID and has only features
    y = df[dv].copy() #copy the dependent variable

    if "ST004D01T" in X.columns:
        X["ST004D01T"] = X["ST004D01T"].map({1: 0, 2: 1})

    print(f"Dropping columns with all missing values...")
    X = X.dropna(axis=1, how="all")  # drop columns where all values are NaN

    print(f"Imputing missing values and scaling features...")
    imputer = SimpleImputer(strategy="median")
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns, index=X.index)

    final_df = X_scaled.copy()
    final_df[dv] = y.values
    return final_df

def save_data(df, output_path):
    print(f"Saving the final processed dataframe...") 
    df.to_pickle(output_path)
    print(f"Saved processed data → {output_path} (shape: {df.shape})")


def preprocess_pisa_data(student_path, school_path, country, predictors, dv, output_path):
    print(f"Preprocessing PISA data for {country}...")
    df = LFM_data(student_path, school_path, country)
    df = process_data(df, predictors, dv)
    save_data(df, output_path)
    print(f"Preprocessing complete.")


In [ ]:
#code/functions for data analysis and model training  



In [8]:
STU_PATH = Path("../data/raw/Student Data.sav") #path to the student data file
SCH_PATH = Path("../data/raw/School Data.sav") #path to the school data file

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet
]

DV = "BEINGBULLIED"

preprocess_pisa_data(STU_PATH, SCH_PATH,"JPN",PREDICTORS, DV, Path("../data/processed/japan_final_2.pkl"))

Preprocessing PISA data for JPN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\japan_final_2.pkl (shape: (6109, 37))
Preprocessing complete.


In [11]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import mean_absolute_error, make_scorer
import numpy as np
import pandas as pd

# Load your processed Japan data
df = pd.read_pickle("../data/processed/japan_final_2.pkl")
df = df.dropna(subset=["BEINGBULLIED"])

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]


xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

# Set up cross-validation
cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

# Use cross_val_score with neg_mean_absolute_error (scikit-learn expects a score, so it's negative MAE)
mae_scores = cross_val_score(xgb, X, y, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1)

# Convert to positive MAE
mae_scores = -mae_scores
mean_mae = np.mean(mae_scores)

print(f"Cross-validated MAE: {np.round(mean_mae, 3)}")


Cross-validated MAE: 0.606


In [12]:
from xgboost import XGBRegressor
import pandas as pd

df = pd.read_pickle("../data/processed/japan_final_2.pkl")  # loads the final cleaned Japan data
df = df.dropna(subset=["BEINGBULLIED"])  # drops rows with missing values in the dependent variable

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]

best_xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

best_xgb.fit(X, y)

importances = best_xgb.feature_importances_  # get feature importances
feat_imp = sorted(zip(X.columns, importances), key=lambda pair: pair[1], reverse=True)

print("Top 10 XGB predictors of BEINGBULLIED (Japan):")
for feat, imp in feat_imp[:10]:
    print(f"  {feat:<12s}: {imp:.4f}")


Top 10 XGB predictors of BEINGBULLIED (Japan):
  BELONG      : 0.0827
  ST004D01T   : 0.0543
  GFOFAIL     : 0.0474
  DISCLIMA    : 0.0395
  EMOSUPS     : 0.0377
  ESCS        : 0.0334
  PERCOMP     : 0.0325
  JOYREAD     : 0.0307
  SCREADCOMP  : 0.0304
  DIRINS      : 0.0289


In [13]:
from xgboost import XGBRegressor
import pandas as pd

df = pd.read_pickle("../data/processed/japan_clean_final.pkl")  # loads the final cleaned Japan data
df = df.dropna(subset=["BEINGBULLIED"])  # drops rows with missing values in the dependent variable

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]

best_xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

best_xgb.fit(X, y)

importances = best_xgb.feature_importances_  # get feature importances
feat_imp = sorted(zip(X.columns, importances), key=lambda pair: pair[1], reverse=True)

print("Top 10 XGB predictors of BEINGBULLIED (Japan):")
for feat, imp in feat_imp[:10]:
    print(f"  {feat:<12s}: {imp:.4f}")


Top 10 XGB predictors of BEINGBULLIED (Japan):
  BELONG      : 0.0895
  ST004D01T   : 0.0644
  GFOFAIL     : 0.0483
  DISCLIMA    : 0.0432
  EMOSUPS     : 0.0380
  SCREADCOMP  : 0.0354
  METASPAM    : 0.0341
  JOYREAD     : 0.0333
  ESCS        : 0.0327
  PERCOMP     : 0.0326


In [14]:
STU_PATH = Path("../data/raw/Student Data.sav") #path to the student data file
SCH_PATH = Path("../data/raw/School Data.sav") #path to the school data file

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet
]

DV = "BEINGBULLIED"

preprocess_pisa_data(STU_PATH, SCH_PATH,"GBR",PREDICTORS, DV, Path("../data/processed/uk_final_2.pkl"))

Preprocessing PISA data for GBR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\uk_final_2.pkl (shape: (13818, 38))
Preprocessing complete.


In [15]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import mean_absolute_error, make_scorer
import numpy as np
import pandas as pd

# Load your processed Japan data
df = pd.read_pickle("../data/processed/uk_final_2.pkl")
df = df.dropna(subset=["BEINGBULLIED"])

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]


xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

# Set up cross-validation
cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

# Use cross_val_score with neg_mean_absolute_error (scikit-learn expects a score, so it's negative MAE)
mae_scores = cross_val_score(xgb, X, y, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1)

# Convert to positive MAE
mae_scores = -mae_scores
mean_mae = np.mean(mae_scores)

print(f"Cross-validated MAE: {np.round(mean_mae, 3)}")


Cross-validated MAE: 0.724


In [16]:
from xgboost import XGBRegressor
import pandas as pd

df = pd.read_pickle("../data/processed/uk_final_2.pkl")  # loads the final cleaned Japan data
df = df.dropna(subset=["BEINGBULLIED"])  # drops rows with missing values in the dependent variable

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]

best_xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

best_xgb.fit(X, y)

importances = best_xgb.feature_importances_  # get feature importances
feat_imp = sorted(zip(X.columns, importances), key=lambda pair: pair[1], reverse=True)

print("Top 10 XGB predictors of BEINGBULLIED (Japan):")
for feat, imp in feat_imp[:10]:
    print(f"  {feat:<12s}: {imp:.4f}")


Top 10 XGB predictors of BEINGBULLIED (Japan):
  BELONG      : 0.2108
  GFOFAIL     : 0.0647
  ST004D01T   : 0.0515
  ST185Q01HA  : 0.0454
  SWBP        : 0.0437
  DISCLIMA    : 0.0405
  PERCOMP     : 0.0321
  EMOSUPS     : 0.0243
  COMPETE     : 0.0242
  PERCOOP     : 0.0222


In [17]:
from xgboost import XGBRegressor
import pandas as pd

df = pd.read_pickle("../data/processed/uk_clean_final.pkl")  # loads the final cleaned Japan data
df = df.dropna(subset=["BEINGBULLIED"])  # drops rows with missing values in the dependent variable

X = df.drop(columns=["BEINGBULLIED"])
y = df["BEINGBULLIED"]

best_xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
)

best_xgb.fit(X, y)

importances = best_xgb.feature_importances_  # get feature importances
feat_imp = sorted(zip(X.columns, importances), key=lambda pair: pair[1], reverse=True)

print("Top 10 XGB predictors of BEINGBULLIED (Japan):")
for feat, imp in feat_imp[:10]:
    print(f"  {feat:<12s}: {imp:.4f}")


Top 10 XGB predictors of BEINGBULLIED (Japan):
  BELONG      : 0.2164
  GFOFAIL     : 0.0666
  ST004D01T   : 0.0583
  DISCLIMA    : 0.0470
  EUDMO       : 0.0381
  PERCOMP     : 0.0355
  EMOSUPS     : 0.0291
  RESILIENCE  : 0.0279
  COMPETE     : 0.0257
  PERCOOP     : 0.0256
